# 14_actuarial_pricing

Actuarial layer added in response to journal review. Three pieces:

1. **Severity.** Link disease onset to annual medical expenditure (from the KHPS
   healthcare-utilization / MS files) and estimate the covariate-adjusted
   incremental cost of a new hypertension / diabetes diagnosis.
2. **Net-premium mispricing.** Combine frequency and severity into a one-period
   net premium and quantify the per-policy mispricing from granting a
   counterfactual-derived credit (~15% expected reduction) when the verified
   effect is null.
3. **Power and equivalence (TOST).** Interpret the null as "excludes a meaningful
   protective effect" rather than "proves zero effect."

Produces Table 17 and Figure 12.

In [1]:
# 14_actuarial_pricing.ipynb  --  cell 1: setup + medical-cost severity input
# Medical cost is loaded from the pre-aggregated parquet if present; otherwise it
# is rebuilt from the raw *_ms.sas7bdat files (large, ~400 MB each).

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf

ROOT = os.path.abspath("..")
DATA = os.path.join(ROOT, "data")
RAW  = os.path.join(DATA, "raw")
FIG  = os.path.join(ROOT, "results", "figures")
TAB  = os.path.join(ROOT, "results", "tables")

COST_PATH = os.path.join(DATA, "medical_costs.parquet")
if os.path.exists(COST_PATH):
    costs = pd.read_parquet(COST_PATH)
    print("loaded medical_costs.parquet:", len(costs), "person-years")
else:
    import pyreadstat
    FILES = [(2019,"a"),(2020,"b"),(2021,"c"),(2022,"d"),(2023,"e"),(2024,"f")]
    def year_cost(pfx, yr):
        f = os.path.join(RAW, f"y{yr}", f"{pfx}_ms.sas7bdat")
        _, meta = pyreadstat.read_sas7bdat(f, metadataonly=True)
        cols = [c for c in ["PIDWON","MEXP3_1","MEXP4_1"] if c in meta.column_names]
        d, _ = pyreadstat.read_sas7bdat(f, usecols=cols)
        for c in ["MEXP3_1","MEXP4_1"]:
            if c not in d.columns: d[c] = 0
            d[c] = d[c].fillna(0).clip(lower=0)
        d["cost"] = d["MEXP3_1"] + d["MEXP4_1"]
        g = d.groupby("PIDWON")["cost"].sum().reset_index(); g["year"] = yr
        return g
    costs = pd.concat([year_cost(p, y) for y, p in FILES], ignore_index=True)
    costs.to_parquet(COST_PATH)
    print("rebuilt medical_costs from MS files:", len(costs))
print("mean annual cost:", f"{costs['cost'].mean():,.0f} KRW")

loaded medical_costs.parquet: 73574 person-years
mean annual cost: 1,102,340 KRW


In [2]:
# cell 2: severity -- incremental annual cost of a new diagnosis (adjusted)
panel = pd.read_parquet(os.path.join(DATA, "khp_panel_long.parquet"))
adult = panel[panel.age >= 19].copy()

PAIRS = [(2019,2020),(2020,2021),(2021,2022),(2022,2023),(2023,2024)]
rows = []
for t0, t1 in PAIRS:
    a = adult[adult.year==t0][["PIDWON","HTN","DM","age","SEX","BMI"]]
    b = adult[adult.year==t1][["PIDWON","HTN","DM"]]
    m = a.merge(b, on="PIDWON", suffixes=("_0","_1"))
    c1 = costs[costs.year==t1][["PIDWON","cost"]]
    m = m.merge(c1, on="PIDWON", how="left"); m["cost"] = m["cost"].fillna(0)
    m["htn_inc"] = ((m["HTN_0"]==0)&(m["HTN_1"]==1)).astype(int)
    m["dm_inc"]  = ((m["DM_0"]==0)&(m["DM_1"]==1)).astype(int)
    rows.append(m)
M = pd.concat(rows, ignore_index=True)

htn = M[M["HTN_0"]==0].dropna(subset=["age","BMI"]).copy()
htn["female"] = (htn["SEX"]==2).astype(int)
sev_htn = smf.ols("cost ~ htn_inc + age + female + BMI", data=htn).fit().params["htn_inc"]

dm = M[M["DM_0"]==0].dropna(subset=["age","BMI"]).copy()
dm["female"] = (dm["SEX"]==2).astype(int)
sev_dm = smf.ols("cost ~ dm_inc + age + female + BMI", data=dm).fit().params["dm_inc"]
print(f"Severity: HTN {sev_htn:,.0f} KRW, DM {sev_dm:,.0f} KRW")

Severity: HTN 354,284 KRW, DM 572,173 KRW


In [3]:
# cell 3: net-premium model and per-policy mispricing
p0 = htn["htn_inc"].mean()
s  = sev_htn
delta_model, delta_true = 0.15, 0.0

pi_standard = p0 * s
pi_credit   = p0 * (1 - delta_model) * s
pi_required = p0 * (1 - delta_true)  * s
mispricing  = pi_required - pi_credit

print(f"Baseline onset p0 = {p0*100:.2f}%")
print(f"Net premium (no credit)    : {pi_standard:,.0f} KRW")
print(f"Net premium (model credit) : {pi_credit:,.0f} KRW")
print(f"Required net premium       : {pi_required:,.0f} KRW")
print(f"Per-policy mispricing      : {mispricing:,.0f} KRW "
      f"({mispricing/pi_required*100:.0f}% under-pricing)")

pd.DataFrame({
    "metric": ["sev_htn","sev_dm","p0_htn","mispricing_per_policy","mispricing_pct"],
    "value":  [round(sev_htn), round(sev_dm), round(p0,4),
               round(mispricing), round(mispricing/pi_required*100,1)],
}).to_csv(os.path.join(TAB, "table17_actuarial.csv"), index=False)

Baseline onset p0 = 2.97%
Net premium (no credit)    : 10,513 KRW
Net premium (model credit) : 8,936 KRW
Required net premium       : 10,513 KRW
Per-policy mispricing      : 1,577 KRW (15% under-pricing)


In [4]:
# cell 4: power and equivalence (TOST)
from statsmodels.stats.power import NormalIndPower

S = pd.read_parquet(os.path.join(DATA, "step4_data.parquet"))
S = S.dropna(subset=["smoke_cur","exer_reg","BMI_0","age"])
n_ach = int(S["achieved"].sum()); n_no = int((S["achieved"]==0).sum())
base_rate = S[S.achieved==0]["incident_t2"].mean()

mde_h = NormalIndPower().solve_power(effect_size=None, nobs1=n_ach, alpha=0.05,
                                     power=0.8, ratio=n_no/n_ach, alternative="two-sided")
print(f"Achievers {n_ach}, non-achievers {n_no}, baseline rate {base_rate*100:.2f}%")
print(f"Minimum detectable effect (Cohen's h) at 80% power: {mde_h:.4f}")
print("Adjusted OR 1.06 (95% CI 0.83-1.35); equivalence margin OR [0.80, 1.25].")
print("Upper CI 1.35 exceeds 1.25 -> strict equivalence not met; data exclude")
print("RR < ~0.75 (a meaningful protective effect of the promised magnitude).")

Achievers 2343, non-achievers 19507, baseline rate 3.05%
Minimum detectable effect (Cohen's h) at 80% power: 0.0613
Adjusted OR 1.06 (95% CI 0.83-1.35); equivalence margin OR [0.80, 1.25].
Upper CI 1.35 exceeds 1.25 -> strict equivalence not met; data exclude
RR < ~0.75 (a meaningful protective effect of the promised magnitude).


In [5]:
# cell 5: Figure 12 -- net premium and mispricing
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3",
                     "grid.color": "0.85", "savefig.dpi": 600})

labels = ["Standard\n(no credit)", "Credited\n(model-expected)", "Required\n(verified)"]
prems  = [pi_standard, pi_credit, pi_required]
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.bar(range(3), prems, color=["0.35","0.7","0.2"], edgecolor="0.15", width=0.6)
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylabel("Net premium (KRW)")
ax.annotate("", xy=(1, pi_credit), xytext=(1, pi_required),
            arrowprops=dict(arrowstyle="<->", color="0.15", lw=1.2))
ax.text(1.15, (pi_credit+pi_required)/2,
        f"mispricing\n{mispricing:,.0f} ({mispricing/pi_required*100:.0f}%)",
        fontsize=8, va="center")
ax.set_ylim(0, max(prems)*1.2)
fig.savefig(os.path.join(FIG, "fig12_pricing_mispricing.png"), dpi=600, bbox_inches="tight")
fig.savefig(os.path.join(FIG, "fig12_pricing_mispricing.pdf"), bbox_inches="tight")
plt.close(fig)
print("Figure 12 saved (png + pdf).")

Figure 12 saved (png + pdf).
